In [3]:
import pandas
import urllib.request
import json
pandas.set_option("display.max_colwidth", None)

In [4]:
API_BASE = "https://api.erg.ic.ac.uk/AirQuality"
API_HELP_PAGE = "https://api.erg.ic.ac.uk/AirQuality/help"
HelpTable = pandas.read_html(API_HELP_PAGE)        # a list of tables
Endpoints = HelpTable[0]                           # the one table
JsonMask = Endpoints["Uri"].str.endswith("/Json")
JsonEndpoints = Endpoints[JsonMask].copy()
assert len(JsonEndpoints) == 36                    # I know there are 36 endpoints that return JSON data
JsonEndpoints[["Uri"]]

,Uri
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json
3,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json
5,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year}/Json
7,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json
9,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json
11,/Annual/MonitoringReport/SiteCode={SiteCode}/Year={Year}/Json
12,/Annualiser/Calculate/year={year}/site={site}/backA={backA}/backB={backB}/backC={backC}/backD={backD}/species={species}/mean={mean}/Json
13,/Annualiser/Sites/year={year}/species={species}/Json
15,/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json
17,/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json


In [5]:
JsonEndpoints["Params"] = JsonEndpoints["Uri"].str.findall(r"\{(\w+)\}")   # the names inside {...} in each Uri
JsonEndpoints[["Uri", "Params"]]

,Uri,Params
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json,[GroupName]
3,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json,"[GroupName, Year]"
5,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year}/Json,"[LocalAuthorityId, Year]"
7,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json,"[SiteCode, StartDate]"
9,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json,"[SiteCode, Year]"
11,/Annual/MonitoringReport/SiteCode={SiteCode}/Year={Year}/Json,"[SiteCode, Year]"
12,/Annualiser/Calculate/year={year}/site={site}/backA={backA}/backB={backB}/backC={backC}/backD={backD}/species={species}/mean={mean}/Json,"[year, site, backA, backB, backC, backD, species, mean]"
13,/Annualiser/Sites/year={year}/species={species}/Json,"[year, species]"
15,/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json,"[GroupName, Date]"
17,/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json,[GroupName]


In [6]:
def CallEndpoint(Row, Values):
    if set(Values) != set(Row["Params"]):                  # names given must match the endpoint's params exactly
        raise ValueError(f"{Row['Uri']} needs {sorted(Row['Params'])}, got {sorted(Values)}")
    Path = Row["Uri"].format(**Values)                     # fill {GroupName} etc. with the values
    with urllib.request.urlopen(API_BASE + Path) as Response:
        return json.loads(Response.read())


Row = JsonEndpoints[JsonEndpoints["Uri"] == "/Information/MonitoringSiteSpecies/GroupName={GroupName}/Json"].iloc[0]
SiteSpecies = CallEndpoint(Row, {"GroupName": "London"})
print(json.dumps(SiteSpecies, indent=2))

{
  "Sites": {
    "Site": [
      {
        "@LocalAuthorityCode": "1",
        "@LocalAuthorityName": "Barking and Dagenham",
        "@SiteCode": "BG3",
        "@SiteName": "Barking and Dagenham - North Street",
        "@SiteType": "Kerbside",
        "@DateClosed": "2011-05-25 00:00:00",
        "@DateOpened": "2007-03-16 00:00:00",
        "@Latitude": "51.540444",
        "@Longitude": "0.074418",
        "@LatitudeWGS84": "6717454.5833",
        "@LongitudeWGS84": "8284.17386585",
        "@DataOwner": "Barking and Dagenham",
        "@DataManager": "King's College London",
        "@SiteLink": "http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG3",
        "Species": {
          "@SpeciesCode": "NO2",
          "@SpeciesDescription": "Nitrogen Dioxide",
          "@DateMeasurementStarted": "2008-01-01 00:00:00",
          "@DateMeasurementFinished": "2011-05-25 00:00:00"
        }
      },
      {
        "@LocalAuthorityCode": "1",
        "@LocalAuthorityName":

In [7]:
GROUP = "London"
SPECIES = ["NO2", "PM25"]
START_DATE = "2026-10-20"

In [8]:
Uri = "/Information/MonitoringSiteSpecies/GroupName={GroupName}/Json"   # copied from your table
Filled = Uri.format(GroupName="London")                                 # swap {GroupName} for London
Filled

'/Information/MonitoringSiteSpecies/GroupName=London/Json'

In [9]:
import json
import urllib.request

API_BASE = "https://api.erg.ic.ac.uk/AirQuality"

Url = API_BASE + Filled                                 # the full web address
with urllib.request.urlopen(Url) as Response:           # visit it
    Data = json.loads(Response.read())                  # turn the JSON text into Python data

Data["Sites"]["Site"]

[{'@LocalAuthorityCode': '1',
  '@LocalAuthorityName': 'Barking and Dagenham',
  '@SiteCode': 'BG3',
  '@SiteName': 'Barking and Dagenham - North Street',
  '@SiteType': 'Kerbside',
  '@DateClosed': '2011-05-25 00:00:00',
  '@DateOpened': '2007-03-16 00:00:00',
  '@Latitude': '51.540444',
  '@Longitude': '0.074418',
  '@LatitudeWGS84': '6717454.5833',
  '@LongitudeWGS84': '8284.17386585',
  '@DataOwner': 'Barking and Dagenham',
  '@DataManager': "King's College London",
  '@SiteLink': 'http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG3',
  'Species': {'@SpeciesCode': 'NO2',
   '@SpeciesDescription': 'Nitrogen Dioxide',
   '@DateMeasurementStarted': '2008-01-01 00:00:00',
   '@DateMeasurementFinished': '2011-05-25 00:00:00'}},
 {'@LocalAuthorityCode': '1',
  '@LocalAuthorityName': 'Barking and Dagenham',
  '@SiteCode': 'BG1',
  '@SiteName': 'Barking and Dagenham - Rush Green',
  '@SiteType': 'Suburban',
  '@DateClosed': '',
  '@DateOpened': '1999-11-02 00:00:00',
  '@Lati

In [10]:
GROUP = "London"
SITES = ["MY1", "GT1"]          # Marylebone Road (kerbside), Station Walk (background)
SPECIES = "NO2"
START_DATE = "2026-10-20"
END_DATE = "2026-10-27"         # the API excludes this day

In [11]:
Uri = "/Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json"
Filled = Uri.format(SiteCode="MY1", SpeciesCode=SPECIES, StartDate=START_DATE, EndDate=END_DATE)

Url = API_BASE + Filled
with urllib.request.urlopen(Url) as Response:
    Readings = json.loads(Response.read())

Readings.keys()

dict_keys(['RawAQData'])

In [12]:
Readings["RawAQData"].keys()

dict_keys(['@SiteCode', '@SpeciesCode', 'Data'])

In [13]:
Readings["RawAQData"]["Data"][0]

{'@MeasurementDateGMT': '2026-10-20 00:00:00', '@Value': ''}

In [ ]:
def GetReadings(Site, Species, Start, End):
    Uri = "/Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json"
    Filled = Uri.format(SiteCode=Site, SpeciesCode=Species, StartDate=Start, EndDate=End)
    with urllib.request.urlopen(API_BASE + Filled) as Response:
        Data = json.loads(Response.read())
    Rows = []
    for Hour in Data["RawAQData"]["Data"]:                  # one entry per hour
        Rows.append({"SiteCode": Site,
                     "SpeciesCode": Species,
                     "MeasuredAt": Hour["@MeasurementDateGMT"],
                     "Value": Hour["@Value"]})
    return pandas.DataFrame(Rows)




In [15]:
GetReadings("MY1", SPECIES, START_DATE, END_DATE)

,SiteCode,SpeciesCode,MeasuredAt,Value
0,MY1,NO2,2026-10-20 00:00:00,
1,MY1,NO2,2026-10-20 01:00:00,45.7
2,MY1,NO2,2026-10-20 02:00:00,42.5
3,MY1,NO2,2026-10-20 03:00:00,32.6
4,MY1,NO2,2026-10-20 04:00:00,34
...,...,...,...,...
163,MY1,NO2,2026-10-26 19:00:00,39.3
164,MY1,NO2,2026-10-26 20:00:00,33.9
165,MY1,NO2,2026-10-26 21:00:00,35.7
166,MY1,NO2,2026-10-26 22:00:00,31.9


In [16]:
AllReadings = pandas.concat([GetReadings(Site, SPECIES, START_DATE, END_DATE) for Site in SITES])

In [ ]:
AllReadings

,SiteCode,SpeciesCode,MeasuredAt,Value
0,MY1,NO2,2026-10-20 00:00:00,
1,MY1,NO2,2026-10-20 01:00:00,45.7
2,MY1,NO2,2026-10-20 02:00:00,42.5
3,MY1,NO2,2026-10-20 03:00:00,32.6
4,MY1,NO2,2026-10-20 04:00:00,34
...,...,...,...,...
163,GT1,NO2,2026-10-26 19:00:00,5.6
164,GT1,NO2,2026-10-26 20:00:00,6.8
165,GT1,NO2,2026-10-26 21:00:00,4.2
166,GT1,NO2,2026-10-26 22:00:00,2.9


In [19]:
Wide = AllReadings.pivot(index="MeasuredAt", columns="SiteCode", values="Value")
Wide

SiteCode,GT1,MY1
MeasuredAt,,
2026-10-20 00:00:00,15.5,
2026-10-20 01:00:00,16.1,45.7
2026-10-20 02:00:00,14.2,42.5
2026-10-20 03:00:00,13.6,32.6
2026-10-20 04:00:00,14.1,34
...,...,...
2026-10-26 19:00:00,5.6,39.3
2026-10-26 20:00:00,6.8,33.9
2026-10-26 21:00:00,4.2,35.7


In [20]:
AllReadings["MeasuredAt"] = pandas.to_datetime(AllReadings["MeasuredAt"])                  # text → real date/time
AllReadings["Value"] = pandas.to_numeric(AllReadings["Value"].replace("", None))           # "" → missing, then text → number
AllReadings.dtypes

SiteCode                  str
SpeciesCode               str
MeasuredAt     datetime64[us]
Value                 float64
dtype: object